In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

print(sys.executable)          # ai-server\.venv 경로여야 정상

load_dotenv(Path("..") / ".env")
KEY = os.getenv("KIPRIS_API_KEY")
print("키 로드:", bool(KEY))

In [ ]:
from pathlib import Path

p = Path("..") / ".env"
print("찾는 경로:", p.resolve())
print("존재 여부:", p.exists())

if p.exists():
    print("---내용---")
    print(p.read_text(encoding="utf-8"))

In [ ]:
print("키 로드:", bool(KEY))

In [ ]:
from pathlib import Path

base = Path("..") / "data" / "trademarks" / "raw"

for p in sorted(base.rglob("*"))[:40]:
    if p.is_file():
        print(p.relative_to(base), f"{p.stat().st_size/1024:.1f}KB")

In [ ]:
base = Path("..") / "data" / "trademarks"

In [ ]:
from pathlib import Path

base = Path("..") / "data" / "trademarks" / "raw"

for p in sorted(base.rglob("*"))[:40]:
    if p.is_file():
        print(p.relative_to(base), f"{p.stat().st_size/1024:.1f}KB")

In [ ]:
from pathlib import Path

base = Path("..") / "data"
print("절대경로:", base.resolve())

for p in sorted(base.rglob("*"))[:50]:
    print(("[D] " if p.is_dir() else "    ") + str(p.relative_to(base)))

In [ ]:
txt_files = sorted(base.rglob("TXT/*"))
print("TXT 파일 수:", len(txt_files))
for f in txt_files[:10]:
    print(f.name, f"{f.stat().st_size/1024:.1f}KB")

In [ ]:
target = txt_files[0]

for enc in ["utf-8", "cp949"]:
    try:
        with open(target, encoding=enc) as f:
            lines = [next(f).rstrip() for _ in range(3)]
        print(f"=== {enc} 성공 ===")
        for l in lines:
            print(l[:300])
        break
    except (UnicodeDecodeError, StopIteration) as e:
        print(f"{enc} 실패: {type(e).__name__}")

In [ ]:
for f in txt_files:
    with open(f, encoding="utf-8") as fh:
        header = fh.readline().rstrip()
    print(f"■ {f.name}")
    print("  ", header.replace("\x02", " | ")[:250])
    print()

In [ ]:
import pandas as pd

def load(name):
    f = next(p for p in txt_files if p.name == name)
    return pd.read_csv(f, sep="\x02", encoding="utf-8", engine="python", dtype=str)

kt10 = load("TB_KT10.txt")
print(kt10.shape)
print(kt10.columns.tolist())

print("\n[최종처분코드명]")
print(kt10["최종처분코드명"].value_counts())

print("\n[상표유형코드명]")
print(kt10["상표유형코드명"].value_counts())

print("\n[상표이미지유무]")
print(kt10["상표이미지유무"].value_counts())

In [ ]:
import pandas as pd

SEP = chr(2)   # \x02

def load(name):
    f = next(p for p in txt_files if p.name == name)
    with open(f, encoding="utf-8") as fh:
        rows = [line.rstrip("\n").split(SEP) for line in fh]
    header, data = rows[0], rows[1:]
    # 행마다 길이가 다를 수 있으니 헤더 길이에 맞춤
    data = [r[:len(header)] + [None] * (len(header) - len(r)) for r in data]
    return pd.DataFrame(data, columns=header, dtype=str)

kt10 = load("TB_KT10.txt")
print(kt10.shape)
print(kt10.columns.tolist()[:12])
kt10.head(3)

In [ ]:
f = next(p for p in txt_files if p.name == "TB_KT10.txt")
with open(f, "rb") as fh:
    raw = fh.read(200)
print(raw)

In [ ]:
import pandas as pd

def load(name, sep="^B"):
    f = next(p for p in txt_files if p.name == name)
    with open(f, encoding="utf-8") as fh:
        rows = [line.rstrip("\n").split(sep) for line in fh]
    header, data = rows[0], rows[1:]
    data = [r[:len(header)] + [None] * (len(header) - len(r)) for r in data]
    return pd.DataFrame(data, columns=header)

kt10 = load("TB_KT10.txt")
print(kt10.shape)
kt10.head(3)

In [ ]:
for col in ["최종처분코드명", "상표유형코드명", "상표이미지유무", "법적상태"]:
    print(f"\n[{col}]")
    print(kt10[col].value_counts(dropna=False))

In [ ]:
print(kt10["상표구분코드명"].value_counts(dropna=False))

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img_dir = next(p for p in base.rglob("IMG") if p.is_dir())

def show(gubun, n=4):
    nos = kt10[kt10["상표구분코드명"] == gubun]["출원번호"].head(n)
    fig, axes = plt.subplots(1, n, figsize=(12, 3))
    for ax, no in zip(axes, nos):
        f = next(img_dir.glob(f"{no}_*"), None)
        if f:
            ax.imshow(Image.open(f)); ax.set_title(no[-6:], fontsize=8)
        ax.axis("off")
    fig.suptitle(gubun)
    plt.show()
for g in ["도형복합", "복합문자", "영문상표"]:
    show(g)

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img_dir = next(p for p in base.rglob("IMG") if p.is_dir())

def show(gubun, n=4):
    nos = kt10[kt10["상표구분코드명"] == gubun]["출원번호"].head(n)
    fig, axes = plt.subplots(1, n, figsize=(12, 3))
    for ax, no in zip(axes, nos):
        f = next(img_dir.glob(f"{no}_*"), None)
        if f:
            ax.imshow(Image.open(f)); ax.set_title(no[-6:], fontsize=8)
        ax.axis("off")
    fig.suptitle(gubun)
    plt.show()

for g in ["도형복합", "복합문자", "영문상표"]:
    show(g)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 모든 IMG 폴더의 이미지를 출원번호로 인덱싱
img_map = {}
for f in base.rglob("IMG/*"):
    no = f.stem.split("_")[0]
    img_map[no] = f

print("인덱싱된 이미지:", len(img_map))

# 매칭 확인
matched = kt10["출원번호"].isin(img_map).sum()
print(f"kt10 {len(kt10)}건 중 이미지 매칭: {matched}건")

In [ ]:
# kt10을 어느 폴더에서 읽었는지 확인
print([p for p in txt_files if p.name == "TB_KT10.txt"])

# 폴더별 이미지 수
from collections import Counter
print(Counter(f.parent.parent.name for f in base.rglob("IMG/*")))

In [ ]:
for d in sorted(base.rglob("DBII_*")):
    if d.is_dir():
        txts = list(d.rglob("TXT/*"))
        imgs = list(d.rglob("IMG/*"))
        print(f"{d.name}: TXT {len(txts)}개, IMG {len(imgs)}개")
        print("   ", [t.name for t in txts][:8])

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 출원번호 → 이미지 경로
img_map = {f.stem.split("_")[0]: f for f in base.rglob("IMG/*")}

def show(gubun, n=4):
    nos = [x for x in kt10[kt10["상표구분코드명"] == gubun]["출원번호"] if x in img_map][:n]
    if not nos:
        print(f"{gubun}: 매칭 이미지 없음")
        return
    fig, axes = plt.subplots(1, len(nos), figsize=(3 * len(nos), 3))
    if len(nos) == 1:
        axes = [axes]
    for ax, no in zip(axes, nos):
        ax.imshow(Image.open(img_map[no]))
        ax.set_title(no[-6:], fontsize=8)
        ax.axis("off")
    fig.suptitle(gubun)
    plt.show()

for g in ["도형복합", "복합문자", "영문상표", "한글상표"]:
    show(g)

In [ ]:
from pathlib import Path
base = Path("..") / "data" / "trademarks"

txts = sorted(base.rglob("TXT/*"))
imgs = list(base.rglob("IMG/*"))

print("TXT:", [f.name for f in txts])
print("IMG:", len(imgs))

In [ ]:
from pathlib import Path

base = Path("..") / "data"
print("절대경로:", base.resolve())

for p in sorted(base.rglob("*"))[:40]:
    mark = "[D]" if p.is_dir() else "   "
    print(mark, p.relative_to(base))

In [ ]:
from pathlib import Path
base = Path("..") / "data" / "trademarks"

txts = sorted(base.rglob("TXT/*"))
imgs = list(base.rglob("IMG/*"))
print("TXT:", [f.name for f in txts])
print("IMG:", len(imgs))


In [ ]:
import pandas as pd

def load(name, sep="^B"):
    f = next(p for p in txts if p.name == name)
    with open(f, encoding="utf-8") as fh:
        rows = [line.rstrip("\n").split(sep) for line in fh]
    header, data = rows[0], rows[1:]
    data = [r[:len(header)] + [None]*(len(header)-len(r)) for r in data]
    return pd.DataFrame(data, columns=header)

kt10 = load("TB_KT10.txt")
print("전체:", len(kt10))
print()
print(kt10["상표구분코드명"].value_counts(dropna=False))
print()
print(kt10["법적상태"].value_counts(dropna=False))

In [ ]:
img_map = {f.stem.split("_")[0]: f for f in imgs}
matched = kt10["출원번호"].isin(img_map).sum()
print(f"{len(kt10)}건 중 이미지 매칭: {matched}건 ({matched/len(kt10)*100:.1f}%)")

# 도형복합 중 이미지 있는 것 = 실제 작업 대상
target = kt10[(kt10["상표구분코드명"] == "도형복합") & (kt10["출원번호"].isin(img_map))]
print("실제 대상:", len(target), "건")


In [ ]:
"""KIPRIS 벌크 데이터 → 유사도 분석용 메타데이터 생성"""
from pathlib import Path
import pandas as pd

BASE = Path(__file__).parent.parent / "data" / "trademarks"
RAW = BASE / "raw"
META = BASE / "meta"
SEP = "^B"


def load(name: str) -> pd.DataFrame:
    f = next(RAW.rglob(f"TXT/{name}"))
    with open(f, encoding="utf-8") as fh:
        rows = [line.rstrip("\n").split(SEP) for line in fh]
    header, data = rows[0], rows[1:]
    data = [r[:len(header)] + [None] * (len(header) - len(r)) for r in data]
    return pd.DataFrame(data, columns=header)


def build():
    # 1) 마스터 — 도형복합만
    kt10 = load("TB_KT10.txt")
    df = kt10[kt10["상표구분코드명"] == "도형복합"].copy()
    df = df[["출원번호", "출원일자", "등록일자", "상표한글명", "상표영문명",
             "상표구분코드명", "법적상태", "존속기간만료일자"]]

    # 2) 비엔나 코드 — 우선순위 1번 + 전체 목록
    kt11 = load("TB_KT11.txt")
    kt11 = kt11.sort_values(["출원번호", "비엔나분류우선순위"])
    primary = kt11.groupby("출원번호")["비엔나분류코드"].first().rename("비엔나_주")
    allcodes = kt11.groupby("출원번호")["비엔나분류코드"].apply(lambda s: "|".join(s)).rename("비엔나_전체")
    df = df.merge(primary, on="출원번호", how="left").merge(allcodes, on="출원번호", how="left")

    # 3) 지정상품 — 류·유사군
    kt15 = load("TB_KT15.txt")
    cls = kt15.groupby("출원번호")["류"].apply(lambda s: "|".join(sorted(set(s)))).rename("류")
    sim = kt15.groupby("출원번호")["유사군"].apply(lambda s: "|".join(sorted(set(s.dropna())))).rename("유사군")
    df = df.merge(cls, on="출원번호", how="left").merge(sim, on="출원번호", how="left")

    # 4) 이미지 경로 매칭
    img_map = {}
    for f in RAW.rglob("IMG/*"):
        img_map.setdefault(f.stem.split("_")[0], f)
    df["이미지경로"] = df["출원번호"].map(lambda x: str(img_map[x].relative_to(BASE)) if x in img_map else None)

    before = len(df)
    df = df[df["이미지경로"].notna()].reset_index(drop=True)

    # 5) 저장
    META.mkdir(parents=True, exist_ok=True)
    out = META / "trademarks.csv"
    df.to_csv(out, index=False, encoding="utf-8-sig")

    print(f"도형복합 {before}건 → 이미지 있는 {len(df)}건 저장")
    print(f"저장 위치: {out}")
    print(f"비엔나 코드 보유: {df['비엔나_주'].notna().sum()}건")
    return df


if __name__ == "__main__":
    build()

In [ ]:
import pandas as pd
df = pd.read_csv("../data/trademarks/meta/trademarks.csv", dtype=str)

print(df.shape)
print(df.columns.tolist())
df.head(3)

In [ ]:
print("[류]")
print(df["류"].value_counts().head())

print("\n[비엔나 주코드 상위]")
print(df["비엔나_주"].value_counts().head(10))

print("\n[법적상태]")
print(df["법적상태"].value_counts())

In [ ]:
from PIL import Image
from collections import Counter
import pandas as pd

df = pd.read_csv("../data/trademarks/meta/trademarks.csv", dtype=str)
BASE = Path("..") / "data" / "trademarks"

modes, sizes = [], []
for p in df["이미지경로"].head(200):
    with Image.open(BASE / p) as im:
        modes.append(im.mode)
        sizes.append(im.size)

print("컬러 모드:", Counter(modes))
print("크기 상위:", Counter(sizes).most_common(5))

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"

FAISS = Path("..") / "data" / "faiss"
emb = np.load(FAISS / "embeddings.npy")
ids = pd.read_csv(FAISS / "ids.csv", dtype=str)["출원번호"].tolist()

# 전체 쌍 유사도 (정규화되어 있으므로 내적 = 코사인)
sim = emb @ emb.T
np.fill_diagonal(sim, np.nan)
flat = sim[~np.isnan(sim)]

print(f"최소 {flat.min():.3f} / 평균 {flat.mean():.3f} / 최대 {flat.max():.3f}")
for q in [50, 90, 95, 99, 99.9]:
    print(f"상위 {100-q:>5.1f}% 지점: {np.percentile(flat, q):.3f}")

plt.hist(flat, bins=60)
plt.title("무작위 상표 쌍 유사도 분포")
plt.show()

In [ ]:
sorted_flat = np.sort(flat)

def to_score(cos):
    return int(np.searchsorted(sorted_flat, cos) / len(sorted_flat) * 100)

from PIL import Image
BASE = Path("..") / "data" / "trademarks"
df = pd.read_csv(BASE / "meta" / "trademarks.csv", dtype=str)
path_map = dict(zip(df["출원번호"], df["이미지경로"]))

iu = np.triu_indices_from(sim, k=1)
vals = sim[iu]
for k in np.argsort(vals)[-8:][::-1]:
    i, j = iu[0][k], iu[1][k]
    fig, ax = plt.subplots(1, 2, figsize=(5, 2.5))
    ax[0].imshow(Image.open(BASE / path_map[ids[i]])); ax[0].axis("off")
    ax[1].imshow(Image.open(BASE / path_map[ids[j]])); ax[1].axis("off")
    fig.suptitle(f"cos {vals[k]:.3f} → {to_score(vals[k])}점")
    plt.show()

In [ ]:
import hashlib

def img_hash(p):
    return hashlib.md5(open(BASE / p, "rb").read()).hexdigest()

df["hash"] = df["이미지경로"].map(img_hash)
dup = df["hash"].duplicated().sum()
print(f"완전 중복: {dup}건 / 고유 이미지: {df['hash'].nunique()}건")

# 거의 동일한 쌍 (0.99 이상)
near = (vals > 0.99).sum()
print(f"cos 0.99 이상 쌍: {near}개")

In [ ]:
def search(query_vec, top_k=3, pool=30, dedup=0.99):
    scores = emb @ query_vec
    order = np.argsort(scores)[::-1][:pool]
    
    results, kept = [], []
    for idx in order:
        # 이미 채택한 결과와 거의 같으면 건너뜀
        if any(emb[idx] @ emb[k] > dedup for k in kept):
            continue
        kept.append(idx)
        results.append({
            "rank": len(results) + 1,
            "출원번호": ids[idx],
            "similarity": to_score(scores[idx]),
        })
        if len(results) == top_k:
            break
    return results

In [ ]:
clean = vals[vals < 0.99]
print(f"제외 전 {len(vals):,}쌍 → 후 {len(clean):,}쌍")
print(f"평균 {clean.mean():.3f}")
for q in [50, 90, 95, 99]:
    print(f"상위 {100-q:>4.1f}%: {np.percentile(clean, q):.3f}")

In [ ]:
from PIL import Image
import torch
from transformers import AutoImageProcessor, AutoModel

processor = AutoImageProcessor.from_pretrained("facebook/dinov2-base")
model = AutoModel.from_pretrained("facebook/dinov2-base").eval()

def embed(path):
    im = Image.open(path).convert("RGB")
    with torch.no_grad():
        v = model(**processor(images=im, return_tensors="pt")).last_hidden_state[:, 0, :].numpy()[0]
    return v / np.linalg.norm(v)

def search(q, top_k=3, pool=30, dedup=0.99):
    scores = emb @ q
    results, kept = [], []
    for idx in np.argsort(scores)[::-1][:pool]:
        if any(emb[idx] @ emb[k] > dedup for k in kept):
            continue
        kept.append(idx)
        row = df.iloc[idx]
        results.append({
            "rank": len(results) + 1,
            "출원번호": ids[idx],
            "name": row["상표한글명"] or row["상표영문명"],
            "similarity": to_score(scores[idx]),
        })
        if len(results) == top_k:
            break
    return results

In [ ]:
import numpy as np, pandas as pd, torch
from pathlib import Path
from PIL import Image
from transformers import AutoImageProcessor, AutoModel

FAISS = Path("..") / "data" / "faiss"
BASE = Path("..") / "data" / "trademarks"

emb = np.load(FAISS / "embeddings.npy")
ids = pd.read_csv(FAISS / "ids.csv", dtype=str)["출원번호"].tolist()
df = pd.read_csv(BASE / "meta" / "trademarks.csv", dtype=str)

sim = emb @ emb.T
np.fill_diagonal(sim, np.nan)
sorted_flat = np.sort(sim[~np.isnan(sim)])

def to_score(cos):
    return int(np.searchsorted(sorted_flat, cos) / len(sorted_flat) * 100)

processor = AutoImageProcessor.from_pretrained("facebook/dinov2-base")
model = AutoModel.from_pretrained("facebook/dinov2-base").eval()

def embed(path):
    im = Image.open(path).convert("RGB")
    with torch.no_grad():
        v = model(**processor(images=im, return_tensors="pt")).last_hidden_state[:, 0, :].numpy()[0]
    return v / np.linalg.norm(v)

def search(q, top_k=3, pool=30, dedup=0.99):
    scores = emb @ q
    results, kept = [], []
    for idx in np.argsort(scores)[::-1][:pool]:
        if any(emb[idx] @ emb[k] > dedup for k in kept):
            continue
        kept.append(idx)
        row = df.iloc[idx]
        results.append({
            "rank": len(results) + 1,
            "출원번호": ids[idx],
            "name": row["상표한글명"] if pd.notna(row["상표한글명"]) else row["상표영문명"],
            "similarity": to_score(scores[idx]),
        })
        if len(results) == top_k:
            break
    return results

print("준비 완료:", emb.shape)

In [ ]:
# 데이터셋 내부 이미지로 자기 자신 찾기 테스트
for r in search(embed(BASE / df.iloc[0]["이미지경로"])):
    print(r)

In [ ]:
def get_name(row):
    for col in ["상표한글명", "상표영문명"]:
        v = row.get(col)
        if pd.notna(v) and str(v).strip():
            return str(v).strip()
    return f"상표 {row['출원번호'][-6:]}"   # 둘 다 없으면 출원번호로

# 결측 규모 확인
blank = df.apply(lambda r: not get_name(r).startswith("상표 "), axis=1)
print(f"이름 있음: {blank.sum()}건 / 없음: {(~blank).sum()}건")

In [ ]:
def search(q, top_k=3, pool=30, dedup=0.99):
    scores = emb @ q
    results, kept = [], []
    for idx in np.argsort(scores)[::-1][:pool]:
        if any(emb[idx] @ emb[k] > dedup for k in kept):
            continue
        kept.append(idx)
        row = df.iloc[idx]
        cls = str(row["류"]).split("|")[0]
        results.append({
            "rank": len(results) + 1,
            "applicationNumber": row["출원번호"],
            "name": get_name(row),
            "category": f"{'화장품' if cls == '03' else cls + '류'} · {row['상표구분코드명']}",
            "similarity": to_score(scores[idx]),
            "imagePath": row["이미지경로"],
        })
        if len(results) == top_k:
            break
    return results


def risk_level(max_sim):
    if max_sim < 30:  return "SAFE"
    if max_sim < 60:  return "MODERATE"
    return "CAUTION"


def analyze(image_path):
    matches = search(embed(image_path))
    max_sim = matches[0]["similarity"] if matches else 0
    return {
        "maxSimilarity": max_sim,
        "riskLevel": risk_level(max_sim),
        "matches": matches,
        "disclaimer": "본 분석은 로고 이미지의 시각적 유사성을 보여주는 참고 자료이며, "
                      "상표 등록 가능 여부나 법적 침해 여부를 판단하지 않습니다.",
    }

In [ ]:
import json
print(json.dumps(analyze(BASE / df.iloc[0]["이미지경로"]), ensure_ascii=False, indent=2))

In [ ]:
"imagePath": row["이미지경로"].replace("\\", "/"),

In [ ]:
df = pd.read_csv(BASE / "meta" / "trademarks.csv", dtype=str)
print(df["이미지경로"].iloc[0])

In [ ]:
test_dir = Path("..") / "data" / "test_logos"
print([f.name for f in test_dir.glob("*")])

In [ ]:
import numpy as np, pandas as pd, torch
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from transformers import AutoImageProcessor, AutoModel

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

FAISS = Path("..") / "data" / "faiss"
BASE = Path("..") / "data" / "trademarks"
test_dir = Path("..") / "data" / "test_logos"

emb = np.load(FAISS / "embeddings.npy")
ids = pd.read_csv(FAISS / "ids.csv", dtype=str)["출원번호"].tolist()
df = pd.read_csv(BASE / "meta" / "trademarks.csv", dtype=str)

sim = emb @ emb.T
np.fill_diagonal(sim, np.nan)
sorted_flat = np.sort(sim[~np.isnan(sim)])

def to_score(cos):
    return int(np.searchsorted(sorted_flat, cos) / len(sorted_flat) * 100)

def get_name(row):
    for col in ["상표한글명", "상표영문명"]:
        v = row.get(col)
        if pd.notna(v) and str(v).strip():
            return str(v).strip()
    return f"상표 {row['출원번호'][-6:]}"

processor = AutoImageProcessor.from_pretrained("facebook/dinov2-base")
model = AutoModel.from_pretrained("facebook/dinov2-base").eval()

def embed(path):
    im = Image.open(path).convert("RGB")
    with torch.no_grad():
        v = model(**processor(images=im, return_tensors="pt")).last_hidden_state[:, 0, :].numpy()[0]
    return v / np.linalg.norm(v)

def search(q, top_k=3, pool=30, dedup=0.99):
    scores = emb @ q
    results, kept = [], []
    for idx in np.argsort(scores)[::-1][:pool]:
        if any(emb[idx] @ emb[k] > dedup for k in kept):
            continue
        kept.append(idx)
        row = df.iloc[idx]
        cls = str(row["류"]).split("|")[0]
        results.append({
            "rank": len(results) + 1,
            "applicationNumber": row["출원번호"],
            "name": get_name(row),
            "category": f"{'화장품' if cls == '03' else cls + '류'} · {row['상표구분코드명']}",
            "similarity": to_score(scores[idx]),
            "imagePath": row["이미지경로"],
        })
        if len(results) == top_k:
            break
    return results

def analyze(image_path):
    matches = search(embed(image_path))
    max_sim = matches[0]["similarity"] if matches else 0
    level = "SAFE" if max_sim < 30 else ("MODERATE" if max_sim < 60 else "CAUTION")
    return {"maxSimilarity": max_sim, "riskLevel": level, "matches": matches}

print("복구 완료:", emb.shape, "| 테스트 이미지:", [f.name for f in test_dir.glob("*")])

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

for f in sorted(test_dir.glob("*")):
    r = analyze(f)
    print(f"\n=== {f.name} ===")
    print(f"최고 유사도 {r['maxSimilarity']} / {r['riskLevel']}")
    
    n = len(r["matches"]) + 1
    fig, ax = plt.subplots(1, n, figsize=(3*n, 3))
    ax[0].imshow(Image.open(f)); ax[0].set_title("생성 로고"); ax[0].axis("off")
    for i, m in enumerate(r["matches"], 1):
        ax[i].imshow(Image.open(BASE / m["imagePath"]))
        ax[i].set_title(f"{m['name'][:12]}\n{m['similarity']}점", fontsize=8)
        ax[i].axis("off")
    plt.show()

In [ ]:
q = embed(next(test_dir.glob("*")))
scores = emb @ q
print(f"최대 {scores.max():.3f} / 평균 {scores.mean():.3f} / 최소 {scores.min():.3f}")
print("상위 10개:", np.sort(scores)[::-1][:10].round(3))

In [ ]:
def analyze(image_path, top_k=3, pool=30, dedup=0.99):
    q = embed(image_path)
    scores = emb @ q
    mu, sd = scores.mean(), scores.std()

    results, kept = [], []
    for idx in np.argsort(scores)[::-1][:pool]:
        if any(emb[idx] @ emb[k] > dedup for k in kept):
            continue
        kept.append(idx)
        row = df.iloc[idx]
        z = (scores[idx] - mu) / sd
        cls = str(row["류"]).split("|")[0]
        results.append({
            "rank": len(results) + 1,
            "applicationNumber": row["출원번호"],
            "name": get_name(row),
            "category": f"{'화장품' if cls == '03' else cls + '류'} · {row['상표구분코드명']}",
            "cos": round(float(scores[idx]), 3),
            "z": round(float(z), 2),
            "similarity": int(min(100, max(0, z * 20))),   # z 5 → 100점
            "imagePath": row["이미지경로"],
        })
        if len(results) == top_k:
            break

    max_sim = results[0]["similarity"] if results else 0
    level = "SAFE" if max_sim < 30 else ("MODERATE" if max_sim < 60 else "CAUTION")
    return {"maxSimilarity": max_sim, "riskLevel": level, "matches": results}

In [ ]:
r = analyze(next(test_dir.glob("*")))
for m in r["matches"]:
    print(f"{m['name'][:15]:15} cos={m['cos']} z={m['z']} → {m['similarity']}점")
print(r["riskLevel"])

In [ ]:
for f in sorted(test_dir.glob("*")):
    r = analyze(f)
    top = r["matches"][0]
    print(f"{f.name[:20]:20} z={top['z']:5.2f} {r['maxSimilarity']:3d}점 {r['riskLevel']:9} ← {top['name'][:15]}")

In [ ]:
for f in sorted(test_dir.glob("*")):
    r = analyze(f)
    top = r["matches"][0]
    print(f"{f.name}")
    print(f"   z={top['z']:.2f} {r['maxSimilarity']}점 {r['riskLevel']} ← {top['name']}\n")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

for f in sorted(test_dir.glob("*")):
    r = analyze(f)
    n = len(r["matches"]) + 1
    fig, ax = plt.subplots(1, n, figsize=(3*n, 3))
    ax[0].imshow(Image.open(f)); ax[0].set_title("쿼리"); ax[0].axis("off")
    for i, m in enumerate(r["matches"], 1):
        ax[i].imshow(Image.open(BASE / m["imagePath"]))
        ax[i].set_title(f"{m['name'][:12]}\nz={m['z']} {m['similarity']}점", fontsize=8)
        ax[i].axis("off")
    fig.suptitle(f"{f.name[:30]} → {r['riskLevel']}")
    plt.show()

In [ ]:
from PIL import Image, ImageDraw, ImageFont

def add_text(src, text, out_path):
    im = Image.open(src).convert("RGB")
    w, h = im.size
    canvas = Image.new("RGB", (w, int(h * 1.3)), "white")
    canvas.paste(im, (0, 0))
    d = ImageDraw.Draw(canvas)
    try:
        font = ImageFont.truetype("C:/Windows/Fonts/times.ttf", int(h * 0.12))
    except:
        font = ImageFont.load_default()
    bbox = d.textbbox((0, 0), text, font=font)
    d.text(((w - bbox[2]) / 2, h * 1.05), text, fill="#333", font=font)
    canvas.save(out_path)

combo_dir = Path("..") / "data" / "test_logos_combo"
combo_dir.mkdir(exist_ok=True)

names = ["AURORA", "BELLA", "LUMIN", "VERTEX"]
for f, nm in zip(sorted(test_dir.glob("*")), names):
    add_text(f, nm, combo_dir / f"{nm}.png")

for f in sorted(combo_dir.glob("*")):
    r = analyze(f)
    top = r["matches"][0]
    print(f"{f.name:12} z={top['z']:5.2f} {r['maxSimilarity']:3d}점 {r['riskLevel']:9} ← {top['name'][:15]}")

In [ ]:
for f in sorted(combo_dir.glob("*")):
    r = analyze(f)
    n = len(r["matches"]) + 1
    fig, ax = plt.subplots(1, n, figsize=(3*n, 3))
    ax[0].imshow(Image.open(f)); ax[0].set_title("쿼리"); ax[0].axis("off")
    for i, m in enumerate(r["matches"], 1):
        ax[i].imshow(Image.open(BASE / m["imagePath"]))
        ax[i].set_title(f"{m['name'][:12]}\nz={m['z']} {m['similarity']}점", fontsize=8)
        ax[i].axis("off")
    fig.suptitle(f"{f.name} → {r['riskLevel']}")
    plt.show()

In [ ]:
# 각 상표가 다른 상표들과 평균적으로 얼마나 유사한지
mean_sim = np.nanmean(sim, axis=1)
top_generic = np.argsort(mean_sim)[::-1][:10]
for i in top_generic:
    print(f"{df.iloc[i]['상표한글명'] or df.iloc[i]['상표영문명']:20} 평균유사도 {mean_sim[i]:.3f}")

In [ ]:
mean_sim = np.nanmean(sim, axis=1)
for i in np.argsort(mean_sim)[::-1][:10]:
    print(f"{get_name(df.iloc[i]):22} {mean_sim[i]:.3f}")

# ELUJAI 위치 확인
rank = {get_name(df.iloc[i]): r for r, i in enumerate(np.argsort(mean_sim)[::-1], 1)}
print("\nELUJAI 순위:", rank.get("ELUJAI"), "/ 670")
print("전체 평균:", mean_sim.mean().round(3))

In [ ]:
half = emb[:335]
q = embed(next(combo_dir.glob("*")))

for name, E in [("335건", half), ("670건", emb)]:
    s = E @ q
    z = (s.max() - s.mean()) / s.std()
    print(f"{name}: 최대 z={z:.2f}")

In [ ]:
kt10 = load("TB_KT10.txt")
print(kt10["상표구분코드명"].value_counts(dropna=False))

In [ ]:
print(df["상표구분코드명"].value_counts(dropna=False))
print("\n명칭 없는 상표:", df["상표한글명"].isna().sum(), "/", len(df))

In [ ]:
from pathlib import Path
import pandas as pd

RAW = Path("..") / "data" / "trademarks" / "raw"

def load_raw(name, sep="^B"):
    f = next(RAW.rglob(f"TXT/{name}"))
    with open(f, encoding="utf-8") as fh:
        rows = [line.rstrip("\n").split(sep) for line in fh]
    header, data = rows[0], rows[1:]
    data = [r[:len(header)] + [None]*(len(header)-len(r)) for r in data]
    return pd.DataFrame(data, columns=header)

kt10 = load_raw("TB_KT10.txt")
print(kt10["상표구분코드명"].value_counts(dropna=False))
print("\n명칭 없는 건:", (kt10["상표한글명"].fillna("").str.strip() == "").sum())

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

RAW = Path("..") / "data" / "trademarks" / "raw"
BASE = Path("..") / "data" / "trademarks"

# 도형상표만 추출
sym = kt10[kt10["상표구분코드명"] == "도형상표"].copy()

# 이미지 경로 매칭
img_map = {}
for f in RAW.rglob("IMG/*"):
    img_map.setdefault(f.stem.split("_")[0], f)

sym["path"] = sym["출원번호"].map(img_map)
sym = sym[sym["path"].notna()].reset_index(drop=True)
print(f"도형상표 {len(sym)}건 (이미지 있는 것)")

# 격자 출력
cols, rows = 8, (len(sym) + 7) // 8
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
for ax in axes.flat:
    ax.axis("off")
for ax, (_, r) in zip(axes.flat, sym.iterrows()):
    ax.imshow(Image.open(r["path"]))
    ax.set_title(r["출원번호"][-6:], fontsize=6)
plt.tight_layout()
plt.show()

In [ ]:
flux_dir = Path("..") / "data" / "flux_logos"

for f in sorted(flux_dir.glob("*")):
    r = analyze(f)
    top = r["matches"][0]
    print(f"{f.name[:22]:22} z={top['z']:5.2f} {r['maxSimilarity']:3d}점 {r['riskLevel']:9} ← {top['name'][:15]}")

In [ ]:
for f in sorted(flux_dir.glob("*")):
    r = analyze(f)
    n = len(r["matches"]) + 1
    fig, ax = plt.subplots(1, n, figsize=(3*n, 3))
    ax[0].imshow(Image.open(f)); ax[0].set_title("FLUX 생성"); ax[0].axis("off")
    for i, m in enumerate(r["matches"], 1):
        ax[i].imshow(Image.open(BASE / m["imagePath"]))
        ax[i].set_title(f"{m['name'][:12]}\n{m['similarity']}점", fontsize=8); ax[i].axis("off")
    fig.suptitle(f"{r['riskLevel']}")
    plt.show()

In [ ]:
from PIL import Image
import numpy as np

src = next(flux_dir.glob("*"))
orig = embed(src)

# 색상 반전 버전
im = Image.open(src).convert("RGB")
inv = Image.fromarray(255 - np.array(im))
inv.save("../data/flux_logos/_tmp_inv.png")
inv_v = embed("../data/flux_logos/_tmp_inv.png")

print(f"원본 vs 색반전: {orig @ inv_v:.3f}")

In [ ]:
def embed_gray(path):
    im = Image.open(path).convert("L").convert("RGB")
    with torch.no_grad():
        v = model(**processor(images=im, return_tensors="pt")).last_hidden_state[:, 0, :].numpy()[0]
    return v / np.linalg.norm(v)

# 색 무관 유사도 확인
g1, g2 = embed_gray(src), embed_gray("../data/flux_logos/_tmp_inv.png")
print(f"그레이스케일 색반전: {g1 @ g2:.3f}")

In [ ]:
# 노트북에서 재측정
emb = np.load(FAISS / "embeddings.npy")
idx = np.random.choice(len(emb), 2000, replace=False)
s = emb[idx] @ emb[idx].T
np.fill_diagonal(s, np.nan)
flat = s[~np.isnan(s)]
print(f"평균 {flat.mean():.3f}")
for q in [50, 90, 95, 99]:
    print(f"상위 {100-q}%: {np.percentile(flat, q):.3f}")

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path

FAISS = Path("..") / "data" / "faiss"
BASE = Path("..") / "data" / "trademarks"

emb = np.load(FAISS / "embeddings.npy")
df = pd.read_csv(BASE / "meta" / "trademarks.csv", dtype=str)
print(f"임베딩 {emb.shape} / 메타 {len(df)}건\n")

# 2000건 샘플로 분포 추정
rng = np.random.default_rng(42)
idx = rng.choice(len(emb), 2000, replace=False)
s = emb[idx] @ emb[idx].T
np.fill_diagonal(s, np.nan)
flat = s[~np.isnan(s)]

print(f"쌍 수: {len(flat):,}")
print(f"평균 {flat.mean():.3f} / 표준편차 {flat.std():.3f}")
for q in [50, 90, 95, 99]:
    print(f"상위 {100-q:>4.1f}%: {np.percentile(flat, q):.3f}")

In [ ]:
import numpy as np, pandas as pd, torch
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from transformers import AutoImageProcessor, AutoModel

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

FAISS = Path("..") / "data" / "faiss"
BASE = Path("..") / "data" / "trademarks"
flux_dir = Path("..") / "data" / "flux_logos"
combo_dir = Path("..") / "data" / "test_logos_combo"

emb = np.load(FAISS / "embeddings.npy")
df = pd.read_csv(BASE / "meta" / "trademarks.csv", dtype=str)

processor = AutoImageProcessor.from_pretrained("facebook/dinov2-base")
model = AutoModel.from_pretrained("facebook/dinov2-base").eval()

def embed(path):
    im = Image.open(path)
    if im.mode in ("RGBA", "LA", "P"):
        im = im.convert("RGBA")
        bg = Image.new("RGBA", im.size, (255, 255, 255, 255))
        im = Image.alpha_composite(bg, im)
    im = im.convert("RGB")
    with torch.no_grad():
        v = model(**processor(images=im, return_tensors="pt")).last_hidden_state[:, 0, :].numpy()[0]

In [ ]:
print(emb.shape, len(df))
print("analyze 준비:", callable(analyze))

In [ ]:
Z_SCALE = 20

def get_name(row):
    for col in ["상표한글명", "상표영문명"]:
        v = row.get(col)
        if pd.notna(v) and str(v).strip():
            return str(v).strip()
    return f"상표 {row['출원번호'][-6:]}"

def embed(path):
    im = Image.open(path)
    if im.mode in ("RGBA", "LA", "P"):
        im = im.convert("RGBA")
        bg = Image.new("RGBA", im.size, (255, 255, 255, 255))
        im = Image.alpha_composite(bg, im)
    im = im.convert("RGB")
    with torch.no_grad():
        v = model(**processor(images=im, return_tensors="pt")).last_hidden_state[:, 0, :].numpy()[0]
    return v / np.linalg.norm(v)

def analyze(path, top_k=3, pool=30, dedup=0.99):
    q = embed(path)
    scores = emb @ q
    mu, sd = scores.mean(), scores.std()
    results, kept = [], []
    for idx in np.argsort(scores)[::-1][:pool]:
        if any(emb[idx] @ emb[k] > dedup for k in kept):
            continue
        kept.append(idx)
        row = df.iloc[idx]
        z = (scores[idx] - mu) / sd
        results.append({
            "rank": len(results) + 1,
            "name": get_name(row),
            "cos": round(float(scores[idx]), 3),
            "z": round(float(z), 2),
            "similarity": int(min(100, max(0, z * Z_SCALE))),
            "imagePath": row["이미지경로"],
        })
        if len(results) == top_k:
            break
    m = results[0]["similarity"] if results else 0
    level = "SAFE" if m < 30 else ("MODERATE" if m < 60 else "CAUTION")
    return {"maxSimilarity": m, "riskLevel": level, "matches": results}

print("함수 준비 완료")

In [ ]:
for f in sorted(flux_dir.glob("*.png")):
    r = analyze(f)
    t = r["matches"][0]
    print(f"{f.name[:24]:24} cos={t['cos']} z={t['z']:5.2f} {r['maxSimilarity']:3d}점 {r['riskLevel']:9} ← {t['name'][:15]}")

In [ ]:
for f in sorted(combo_dir.glob("*.png")):
    r = analyze(f)
    t = r["matches"][0]
    print(f"{f.name[:12]:12} z={t['z']:5.2f} {r['maxSimilarity']:3d}점 {r['riskLevel']:9} ← {t['name'][:15]}")

In [ ]:
for f in sorted(flux_dir.glob("*.png")):
    if "_tmp" in f.name: continue
    q = embed(f)
    s = emb @ q
    print(f"{f.name[:24]:24} max={s.max():.3f} mean={s.mean():.3f} sd={s.std():.3f} z={(s.max()-s.mean())/s.std():.2f}")

In [ ]:
for f in sorted(flux_dir.glob("*.png")):
    r = analyze(f)
    n = len(r["matches"]) + 1
    fig, ax = plt.subplots(1, n, figsize=(3*n, 3))
    ax[0].imshow(Image.open(f)); ax[0].set_title("쿼리"); ax[0].axis("off")
    for i, m in enumerate(r["matches"], 1):
        ax[i].imshow(Image.open(BASE / m["imagePath"]))
        ax[i].set_title(f"{m['name'][:12]}\n{m['similarity']}점", fontsize=8); ax[i].axis("off")
    fig.suptitle(r["riskLevel"]); plt.show()

In [ ]:
# 데이터셋 상표 200개를 쿼리로 넣어 1위 z 분포 구하기 (자기 자신 제외)
rng = np.random.default_rng(0)
sample = rng.choice(len(emb), 200, replace=False)

zs = []
for i in sample:
    s = emb @ emb[i]
    s[i] = -1                      # 자기 자신 제외
    mask = s < 0.99                # 중복 상표 제외
    s = np.where(mask, s, -1)
    z = (s.max() - s.mean()) / s.std()
    zs.append(z)

zs = np.array(zs)
print(f"실제 상표 간 1위 z — 평균 {zs.mean():.2f}")
for q in [50, 75, 90, 95, 99]:
    print(f"  상위 {100-q:>4.1f}%: {np.percentile(zs, q):.2f}")

In [ ]:
Z_SLOPE, Z_INTERCEPT = 13.7, -7.3

def to_score(z):
    return int(min(100, max(0, z * Z_SLOPE + Z_INTERCEPT)))

for f in sorted(flux_dir.glob("*.png")):
    r = analyze(f)
    t = r["matches"][0]
    new = to_score(t["z"])
    level = "SAFE" if new < 30 else ("MODERATE" if new < 60 else "CAUTION")
    print(f"{f.name[:22]:22} z={t['z']:5.2f}  {r['maxSimilarity']:3d}점 → {new:3d}점  {level}")

In [ ]:
import sys
sys.path.insert(0, "../logo_ai_server")
from logo_composer import compose_logo
from PIL import Image

symbol = Image.open(next(flux_dir.glob("*.png")))

# 유사도 검증용은 반드시 흰 배경
out = compose_logo(symbol, "AURORA", style="혼합형", background="white")
out.save("../data/flux_logos/composed_test.png")
out

In [ ]:
import inspect
import logo_composer
print(inspect.signature(logo_composer.compose_logo))
print()
print([n for n in dir(logo_composer) if not n.startswith("_")])

In [ ]:
for n in ["compose_final_logo", "compose_logo_with_text", "compose_logos_with_text"]:
    print(n, inspect.signature(getattr(logo_composer, n)))
    print("  ", (getattr(logo_composer, n).__doc__ or "").strip().split("\n")[0])
    print()

In [ ]:
out = compose_logo(symbol, "AURORA", style="혼합형")
out.save("../data/flux_logos/composed_test.png")
out

In [ ]:
r = analyze("../data/flux_logos/composed_test.png")
print(r["maxSimilarity"], r["riskLevel"])
for m in r["matches"]:
    print(f"  {m['name'][:20]:20} {m['similarity']}점")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def show(path, top_k=3):
    r = analyze(path, top_k=top_k)
    n = len(r["matches"]) + 1
    fig, ax = plt.subplots(1, n, figsize=(3.2 * n, 3.4))
    if n == 1:
        ax = [ax]

    ax[0].imshow(Image.open(path).convert("RGB"))
    ax[0].set_title("쿼리", fontsize=11)
    ax[0].axis("off")

    for i, m in enumerate(r["matches"], 1):
        ax[i].imshow(Image.open(BASE / m["imagePath"]).convert("RGB"))
        ax[i].set_title(f"{m['name'][:14]}\n{m['similarity']}점 (z={m['z']})", fontsize=9)
        ax[i].axis("off")

    fig.suptitle(f"{Path(path).name} → {r['maxSimilarity']}점 {r['riskLevel']}", fontsize=11)
    plt.tight_layout()
    plt.show()
    return r


In [ ]:
show("../data/flux_logos/composed_test.png", top_k=5)

In [ ]:
import base64, io, json, os, requests
from PIL import Image
from dotenv import load_dotenv

load_dotenv("../.env")
GEMINI_KEY = os.getenv("GEMINI_API_KEY")
MODEL = "gemini-flash-latest"   # AI Studio에서 정확한 모델명 확인 후 교체

PROMPT = """첫 번째 이미지는 사용자가 생성한 로고이고, 나머지는 기존 등록 상표입니다.
각 등록 상표가 생성 로고와 시각적으로 어떤 점이 닮았는지 한 문장씩 설명하세요.

규칙:
- 도형의 형태·구도·배치 중심으로 설명 (색상은 부수적)
- 40자 이내, "~해요" 체
- 닮은 점이 뚜렷하지 않으면 그렇게 쓰세요
- 법적 판단(침해 여부, 등록 가능성)은 절대 언급 금지

JSON 배열로만 답하세요: ["설명1", "설명2", "설명3"]"""


def _part(img_src, max_size=384):
    im = Image.open(img_src).convert("RGB")
    im.thumbnail((max_size, max_size))
    buf = io.BytesIO()
    im.save(buf, format="JPEG", quality=85)
    return {"inline_data": {"mime_type": "image/jpeg",
                            "data": base64.b64encode(buf.getvalue()).decode()}}


def generate_notes(query_path, matches):
    parts = [{"text": PROMPT}, _part(query_path)]
    parts += [_part(BASE / m["imagePath"]) for m in matches]

    url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"
    try:
        res = requests.post(
            url,
            params={"key": GEMINI_KEY},
            json={"contents": [{"parts": parts}],
                  "generationConfig": {"temperature": 0.3, "maxOutputTokens": 300}},
            timeout=20,
        )
        res.raise_for_status()
        text = res.json()["candidates"][0]["content"]["parts"][0]["text"]
        text = text.replace("```json", "").replace("```", "").strip()
        notes = json.loads(text)
        return notes + [""] * (len(matches) - len(notes))
    except Exception as e:
        print("note 생성 실패:", e)
        return [""] * len(matches)

In [ ]:
r = analyze("../data/flux_logos/composed_test.png")
notes = generate_notes("../data/flux_logos/composed_test.png", r["matches"])

for m, note in zip(r["matches"], notes):
    print(f"{m['name'][:15]:15} {m['similarity']}점")
    print(f"   {note}\n")

In [ ]:
MODEL = "gemini-3-flash-preview"

In [ ]:
load_dotenv("../.env", override=True)
GEMINI_KEY = os.getenv("GEMINI_API_KEY")
print("키 로드:", bool(GEMINI_KEY))

In [ ]:
r = analyze("../data/flux_logos/composed_test.png")
notes = generate_notes("../data/flux_logos/composed_test.png", r["matches"])

for m, note in zip(r["matches"], notes):
    print(f"{m['name'][:15]:15} {m['similarity']}점")
    print(f"   {note}\n")

In [ ]:
import time

def generate_notes(query_path, matches):
    parts = [{"text": PROMPT}, _part(query_path)]
    parts += [_part(BASE / m["imagePath"]) for m in matches]

    url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"
    payload = {
        "contents": [{"parts": parts}],
        "generationConfig": {"temperature": 0.3, "maxOutputTokens": 300},
    }
    headers = {"x-goog-api-key": GEMINI_KEY}

    for attempt in range(3):
        try:
            res = requests.post(url, headers=headers, json=payload, timeout=30)
            if res.status_code in (429, 500, 503):
                time.sleep(2 ** attempt)
                continue
            res.raise_for_status()
            text = res.json()["candidates"][0]["content"]["parts"][0]["text"]
            text = text.replace("```json", "").replace("```", "").strip()
            notes = json.loads(text)
            return notes + [""] * (len(matches) - len(notes))
        except Exception as e:
            print(f"시도 {attempt + 1} 실패: {type(e).__name__}")
            time.sleep(2 ** attempt)

    return [""] * len(matches)

In [ ]:
MODEL = "gemini-flash-latest"

In [ ]:
r = analyze("../data/flux_logos/composed_test.png")
notes = generate_notes("../data/flux_logos/composed_test.png", r["matches"])

for m, note in zip(r["matches"], notes):
    print(f"{m['name'][:15]:15} {m['similarity']}점")
    print(f"   {note}\n")

In [ ]:
parts = [{"text": PROMPT}, _part("../data/flux_logos/composed_test.png")]
parts += [_part(BASE / m["imagePath"]) for m in r["matches"]]

res = requests.post(
    f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent",
    headers={"x-goog-api-key": GEMINI_KEY},
    json={"contents": [{"parts": parts}],
          "generationConfig": {"temperature": 0.3, "maxOutputTokens": 300}},
    timeout=30,
)
print(res.status_code)
print(res.text[:1500])

In [ ]:
print(GEMINI_KEY[:4], len(GEMINI_KEY))

In [ ]:
def generate_notes(query_path, matches):
    parts = [{"text": PROMPT}, _part(query_path)]
    parts += [_part(BASE / m["imagePath"]) for m in matches]

    url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"
    payload = {
        "contents": [{"parts": parts}],
        "generationConfig": {"temperature": 0.3, "maxOutputTokens": 300},
    }

    for attempt in range(3):
        try:
            res = requests.post(url, params={"key": GEMINI_KEY}, json=payload, timeout=30)
            if res.status_code in (429, 500, 503):
                print(f"시도 {attempt + 1}: HTTP {res.status_code} 재시도")
                time.sleep(2 ** attempt)
                continue
            if res.status_code != 200:
                print(f"HTTP {res.status_code}")
                print(res.text[:500])          # 본문만 — URL 없음
                return [""] * len(matches)

            text = res.json()["candidates"][0]["content"]["parts"][0]["text"]
            text = text.replace("```json", "").replace("```", "").strip()
            notes = json.loads(text)
            return notes + [""] * (len(matches) - len(notes))
        except Exception as e:
            print(f"시도 {attempt + 1} 실패: {type(e).__name__}")
            time.sleep(2 ** attempt)

    return [""] * len(matches)

In [ ]:
load_dotenv("../.env", override=True)
GEMINI_KEY = os.getenv("GEMINI_API_KEY")
print("길이:", len(GEMINI_KEY))

In [ ]:
r = analyze("../data/flux_logos/composed_test.png")
notes = generate_notes("../data/flux_logos/composed_test.png", r["matches"])

for m, note in zip(r["matches"], notes):
    print(f"{m['name'][:15]:15} {m['similarity']}점")
    print(f"   {note}\n")

In [ ]:
load_dotenv("../.env", override=True)
GEMINI_KEY = os.getenv("GEMINI_API_KEY")
print("길이:", len(GEMINI_KEY))
print("공백/줄바꿈 포함:", GEMINI_KEY != GEMINI_KEY.strip())

In [ ]:
res = requests.get("https://generativelanguage.googleapis.com/v1beta/models",
                   params={"key": GEMINI_KEY}, timeout=15)
print(res.status_code)
print(res.text[:400])

In [ ]:
res = requests.post(
    f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent",
    headers={
        "Content-Type": "application/json",
        "X-goog-api-key": GEMINI_KEY,
    },
    json={"contents": [{"parts": [{"text": "Explain how AI works in a few words"}]}]},
    timeout=20,
)
print(res.status_code)
print(res.text[:500])

In [ ]:
res = requests.get(
    "https://generativelanguage.googleapis.com/v1beta/models",
    headers={"X-goog-api-key": GEMINI_KEY},
    timeout=15,
)
print(res.status_code)
print(res.text[:400])

In [ ]:
load_dotenv("../.env", override=True)
GEMINI_KEY = os.getenv("GEMINI_API_KEY")

res = requests.get("https://generativelanguage.googleapis.com/v1beta/models",
                   headers={"X-goog-api-key": GEMINI_KEY}, timeout=15)
print(res.status_code)
print(res.text[:300])

In [ ]:
r = analyze("../data/flux_logos/composed_test.png")
notes = generate_notes("../data/flux_logos/composed_test.png", r["matches"])

for m, note in zip(r["matches"], notes):
    print(f"{m['name'][:15]:15} {m['similarity']}점")
    print(f"   {note}\n")

In [ ]:
parts = [{"text": PROMPT}, _part("../data/flux_logos/composed_test.png")]
parts += [_part(BASE / m["imagePath"]) for m in r["matches"]]

res = requests.post(
    f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent",
    headers={"Content-Type": "application/json", "X-goog-api-key": GEMINI_KEY},
    json={"contents": [{"parts": parts}],
          "generationConfig": {"temperature": 0.3, "maxOutputTokens": 300}},
    timeout=30,
)
print(res.status_code)
print(res.text[:1500])

In [ ]:
def generate_notes(query_path, matches):
    parts = [{"text": PROMPT}, _part(query_path)]
    parts += [_part(BASE / m["imagePath"]) for m in matches]

    url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"
    payload = {
        "contents": [{"parts": parts}],
        "generationConfig": {
            "temperature": 0.3,
            "maxOutputTokens": 2000,
            "responseMimeType": "application/json",
            "responseSchema": {"type": "ARRAY", "items": {"type": "STRING"}},
        },
    }
    headers = {"Content-Type": "application/json", "X-goog-api-key": GEMINI_KEY}

    for attempt in range(3):
        try:
            res = requests.post(url, headers=headers, json=payload, timeout=60)
            if res.status_code in (429, 500, 503):
                print(f"시도 {attempt + 1}: HTTP {res.status_code} 재시도")
                time.sleep(2 ** attempt)
                continue
            if res.status_code != 200:
                print(f"HTTP {res.status_code}")
                print(res.text[:500])
                return [""] * len(matches)

            out = res.json()["candidates"][0]["content"]["parts"]
            text = "".join(p.get("text", "") for p in out)
            text = text.replace("```json", "").replace("```", "").strip()
            notes = json.loads(text)
            return notes + [""] * (len(matches) - len(notes))
        except Exception as e:
            print(f"시도 {attempt + 1} 실패: {type(e).__name__}: {e}")
            time.sleep(2 ** attempt)

    return [""] * len(matches)

In [ ]:
r = analyze("../data/flux_logos/composed_test.png")
notes = generate_notes("../data/flux_logos/composed_test.png", r["matches"])

for m, note in zip(r["matches"], notes):
    print(f"{m['name'][:15]:15} {m['similarity']}점")
    print(f"   {note}\n")